# Module 2: Taking Advantage of Iceberg Tables

This notebook contains code examples for Module 2 videos.

## Setup

Initialize Spark session for Module 2 examples.

In [ ]:
# Initialize Spark session for Module 2 examples

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("FirstStepNotebook") \
    .getOrCreate()


In [ ]:
# Configure MinIO client for S3 operations

import boto3
from botocore.client import Config

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

## Video 1: Moving Existing Data to Iceberg


In [ ]:
# Clean up any November data to ensure it's not present during snapshot

# Delete November file from trips_snapshot/data if it exists
try:
    s3_client.delete_object(Bucket='warehouse', Key='taxi/trips_snapshot/data/yellow_tripdata_2023-11.parquet')
    print("Deleted November data from trips_snapshot/data/")
except:
    print("November data not present in trips_snapshot/data/")

### View Raw Data Files

[Open trips_snapshot in MinIO](http://localhost:9001/browser/warehouse/taxi/trips_snapshot/)

Login: `admin` / `password`

The table's metadata is in `/metadata/` and data files are in `/data/` - both under the same table location.

In [ ]:
# Create temporary table in session catalog, then snapshot it to create Iceberg metadata

# Drop existing snapshot table if it exists
spark.sql("DROP TABLE IF EXISTS polaris.taxi.trips_snapshot")

# Switch to session catalog and create a table that references the Parquet files
spark.sql("USE spark_catalog")

spark.sql("""
    CREATE TABLE IF NOT EXISTS default.temp_parquet_source
    USING parquet
    LOCATION 's3a://warehouse/taxi/trips_snapshot/data/'
""")

# Snapshot with location at taxi/trips_snapshot - data files are already in place
spark.sql("""
    CALL polaris.system.snapshot(
        source_table => 'default.temp_parquet_source',
        table => 'polaris.taxi.trips_snapshot',
        location => 's3a://warehouse/taxi/trips_snapshot'
    )
""").show()

# Switch back to polaris catalog
spark.sql("USE polaris")

# Clean up the temporary table
spark.sql("DROP TABLE IF EXISTS spark_catalog.default.temp_parquet_source")

### View Snapshot Table in MinIO

[Open trips_snapshot in MinIO](http://localhost:9001/browser/warehouse/taxi/trips_snapshot/)

Login: `admin` / `password`

The table's metadata is in `/metadata/` and data files are in `/data/` - both under the same table location.

In [ ]:
# Verify the snapshot created successfully and view file metadata

spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_snapshot
""").show()

spark.sql("""
    SELECT 
        file_path,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
        record_count
    FROM polaris.taxi.trips_snapshot.files
""").show(truncate=False)

### Download an additional month of taxi data

In [ ]:
# Download November 2023 taxi data for add_files demonstration

import urllib.request
import os

# Download November data directly into the table's directory
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-11.parquet"
filename = "yellow_tripdata_2023-11.parquet"
local_path = f"/tmp/{filename}"
s3_key = f"taxi/trips_snapshot/data/{filename}"

print(f"Downloading {filename}...")
urllib.request.urlretrieve(url, local_path)

print(f"Uploading to MinIO: {s3_key}")
s3_client.upload_file(local_path, 'warehouse', s3_key)

os.remove(local_path)
print(f"Uploaded {filename} to s3://warehouse/taxi/trips_snapshot/data/")

In [ ]:
# Use add_files to incrementally add new Parquet files to an existing Iceberg table

spark.sql("""
    CALL polaris.system.add_files(
        table => 'polaris.taxi.trips_snapshot',
        source_table => 'parquet.`s3a://warehouse/taxi/trips_snapshot/data/yellow_tripdata_2023-11.parquet`'
    )
""").show()

In [ ]:
# Check the files metadata table to see the new files

spark.sql("""
    SELECT COUNT(*) FROM polaris.taxi.trips_snapshot.files
""").show()


In [ ]:
# Verify the incremental file additions worked correctly

spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_snapshot
""").show()

### Reserialization with CTAS

When data isn't in a compatible format (CSV, JSON, etc.) or you need to transform during migration, use CREATE TABLE AS SELECT. This is compute-intensive but provides complete flexibility.

In [ ]:
import tempfile
import os

csv_data = """VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,total_amount
1,2023-06-01 10:15:00,2023-06-01 10:30:00,2,3.5,15.0,20.5
2,2023-06-01 11:20:00,2023-06-01 11:45:00,1,5.2,22.0,28.0
1,2023-06-01 14:30:00,2023-06-01 15:00:00,3,7.8,30.0,38.5
"""

temp_dir = "/tmp/csv_demo"
os.makedirs(temp_dir, exist_ok=True)
csv_path = os.path.join(temp_dir, "sample_trips.csv")

with open(csv_path, 'w') as f:
    f.write(csv_data)

In [ ]:
# Migrate CSV data to Iceberg using CTAS with proper schema transformation

# First, read the CSV with headers
csv_df = spark.read.format("csv") \
    .option("header", "true") \
    .load("file:///tmp/csv_demo/sample_trips.csv")

# Create temporary view for CTAS
csv_df.createOrReplaceTempView("csv_source")

# Now create the Iceberg table from the CSV
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_from_csv
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3'
    )
    AS 
    SELECT 
        CAST(VendorID AS INT) as VendorID,
        CAST(tpep_pickup_datetime AS TIMESTAMP) as tpep_pickup_datetime,
        CAST(tpep_dropoff_datetime AS TIMESTAMP) as tpep_dropoff_datetime,
        CAST(passenger_count AS INT) as passenger_count,
        CAST(trip_distance AS DOUBLE) as trip_distance,
        CAST(fare_amount AS DOUBLE) as fare_amount,
        CAST(total_amount AS DOUBLE) as total_amount
    FROM csv_source
""")

In [ ]:
# View our CSV Data now in an Iceberg Table

spark.sql("SELECT * FROM polaris.taxi.trips_from_csv").show()

## Video 2: Git-Like Features with Write-Audit-Publish and Branching and Tagging


In [ ]:
# Create trips_by_month table with WAP enabled for Write-Audit-Publish pattern
spark.sql(""" DROP TABLE IF EXISTS polaris.taxi.trips_by_month """)
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_month
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.wap.enabled' = 'true'
    )
    AS
    SELECT * FROM parquet.`s3a://warehouse/raw/`
""")
spark.sql("SELECT COUNT(*) from polaris.taxi.trips_by_month").show()

In [ ]:
# Add a new column to the schema
# In this example, we'll add a correct_fare column (half of the incorrect total_amount)

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    ADD COLUMN correct_fare DOUBLE
""")

In [ ]:
# Enable Write-Audit-Publish (WAP) and perform UPDATE operation
# Set a wap.id to identify this staged write operation
# The changes will be written but not immediately visible to normal queries

spark.conf.set("spark.wap.id", "wap-demo-001")

spark.sql("""
    UPDATE polaris.taxi.trips_by_month
    SET correct_fare = total_amount / 2.0
""")

# Unset the wap.id to ensure subsequent operations are not part of this WAP operation
spark.conf.unset("spark.wap.id")

In [ ]:
# Query the production table WITHOUT the WAP snapshot - correct_fare should be NULL
print("\nData in production table (WITHOUT WAP changes - correct_fare is NULL):")
spark.sql("""
    SELECT tpep_pickup_datetime, total_amount, correct_fare
    FROM polaris.taxi.trips_by_month
    LIMIT 10
""").show()

In [ ]:
# Find the snapshot_id that corresponds to our wap.id
# This snapshot contains the staged changes

from pyspark.sql.functions import col

wap_snapshot_id = spark.sql("""
    SELECT snapshot_id 
    FROM polaris.taxi.trips_by_month.snapshots
    WHERE summary['wap.id'] = 'wap-demo-001'
""").collect()[0]['snapshot_id']

print(f"WAP Snapshot ID: {wap_snapshot_id}")

# Query the WAP snapshot to audit the changes before publishing
# Use VERSION AS OF syntax with snapshot ID to see the staged data
print("\nData in WAP snapshot (with changes):")
spark.sql(f"""
    SELECT tpep_pickup_datetime, total_amount, correct_fare
    FROM polaris.taxi.trips_by_month VERSION AS OF {wap_snapshot_id}
    LIMIT 10
""").show()


In [ ]:
# Option 1: Drop the WAP snapshot if validation fails
# Use expire_snapshots to remove the WAP snapshot without publishing it
# This discards the staged changes as if the operation never happened

# spark.sql(f"""
#     CALL polaris.system.expire_snapshots(
#         table => 'polaris.taxi.trips_by_month',
#         snapshot_ids => ARRAY({wap_snapshot_id})
#     )
# """)

In [ ]:
# Option 2: Publish the WAP snapshot to make it visible to all readers
# This promotes the changes from staging to production

spark.sql("""
    CALL polaris.system.publish_changes(
        table => 'polaris.taxi.trips_by_month',
        wap_id => 'wap-demo-001'
    )
""")

spark.sql("""
    SELECT tpep_pickup_datetime, total_amount, correct_fare
    FROM polaris.taxi.trips_by_month
    LIMIT 10
""").show()

In [ ]:
# Drop the correct_fare column before branching
# This ensures the branch has the same schema as the source parquet files

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    DROP COLUMN correct_fare
""")

In [ ]:
# Create a branch for experimentation
# This creates an isolated fork of the table history

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    CREATE BRANCH experiment_branch
""")

In [ ]:
# Insert data to the branch only
# Main branch remains unaffected

# Insert a few days of data from October into the branch
spark.sql("""
    INSERT INTO polaris.taxi.trips_by_month.branch_experiment_branch
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-10.parquet`
    WHERE tpep_pickup_datetime >= '2023-10-01'
      AND tpep_pickup_datetime < '2023-10-04'
""")

In [ ]:
# Query the branch to verify changes - shows the 3 new rows
print("Branch (experiment_branch) with new data:")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month.branch_experiment_branch
""").show()

# Query the main branch - should not have the new rows
print("\nMain branch (unchanged):")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

In [ ]:
# Merge the branch back to main
# All commits from the branch become part of main's history

spark.sql("""
    CALL polaris.system.fast_forward(
        table => 'polaris.taxi.trips_by_month',
        branch => 'main',
        to => 'experiment_branch'
    )
""").show()

In [ ]:
# Query the branch to verify changes - shows the 3 new rows
print("Branch (experiment_branch) :")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month.branch_experiment_branch
""").show()

# Query the main branch 
print("\nMain branch:")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

In [ ]:
# Create a tag to mark an important snapshot
# Tags give snapshots memorable names instead of using snapshot IDs

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    CREATE TAG end_of_quarter
""")

In [ ]:
# Query using the tag
spark.sql("""
    SELECT COUNT(*) as trip_count
    FROM polaris.taxi.trips_by_month.tag_end_of_quarter
""").show()

In [ ]:
# Rollback to a previous snapshot for disaster recovery
# First, let's see the snapshot history

spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation
    FROM polaris.taxi.trips_by_month.snapshots
    ORDER BY committed_at DESC
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Rollback to the snapshot before the branch merge
# Get the second-most recent snapshot (before the merge)

from pyspark.sql.functions import col

# Get the snapshot before the current one
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM polaris.taxi.trips_by_month.snapshots
    ORDER BY committed_at DESC
    LIMIT 2
""").collect()

previous_snapshot_id = snapshots[1]['snapshot_id']
print(f"Rolling back to snapshot: {previous_snapshot_id}")
print(f"Operation: {snapshots[1]['operation']}")

# Show Current Row Count
print("\nCurrent Row Count:")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

# Perform the rollback
spark.sql(f"""
    CALL polaris.system.rollback_to_snapshot(
        table => 'polaris.taxi.trips_by_month',
        snapshot_id => {previous_snapshot_id}
    )
""").show()

# Verify we're back to the original row count (before the merge)
print("\nRow count after rollback (should match original count before merge):")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

## Video 3: Schema Evolution for Iceberg Tables


In [ ]:
# Setup: Create a fresh trips_by_day table for schema evolution demonstrations
spark.sql("DROP TABLE IF EXISTS polaris.taxi.trips_by_day")

spark.sql("""
    CREATE TABLE polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS
    SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

In [ ]:
# Add a column to the table
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ADD COLUMN tip_percentage DOUBLE
""")

In [ ]:
# Rename a column
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    RENAME COLUMN tip_percentage TO tip_pct
""")

In [ ]:
# Drop a column
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    DROP COLUMN tip_pct
""")

In [ ]:
# View the table schema to see field IDs
spark.sql("""
    DESCRIBE EXTENDED polaris.taxi.trips_by_day
""").show(truncate=False)

In [ ]:
# Add a column with both initial default and write default
# Initial default: retroactively applies to all existing data
# Write default: applies to future writes when value not provided

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ADD COLUMN discount_applied BOOLEAN 
    DEFAULT false
""")

In [ ]:
# Query to verify the default value is applied to existing records
spark.sql("""
    SELECT 
        tpep_pickup_datetime,
        total_amount,
        discount_applied
    FROM polaris.taxi.trips_by_day
    LIMIT 10
""").show()

In [ ]:
# Change the write default for future inserts
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ALTER COLUMN discount_applied SET DEFAULT true
""")

## Video 4: Partition Evolution for Iceberg Tables

In [ ]:
# Example: Change partition spec on an existing table
# Start with an unpartitioned table, then add day partitioning
spark.sql(""" DROP TABLE IF EXISTS polaris.taxi.trips_evolving """)
# First, create an unpartitioned table if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_evolving
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '3'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

In [ ]:
# View current partition spec
spark.sql("""
    DESCRIBE TABLE EXTENDED polaris.taxi.trips_evolving
""").show(100)

In [ ]:
# Add day partitioning to the table
# Old files remain unchanged; new writes will use day partitioning

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    ADD PARTITION FIELD days(tpep_pickup_datetime)
""")

In [ ]:
# Insert new data - it will use the new partition spec
spark.sql("""
    INSERT INTO polaris.taxi.trips_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

In [ ]:
# View partitions now - mix of unpartitioned and day-partitioned data
spark.sql("""
    SELECT partition FROM polaris.taxi.trips_evolving.partitions
""").show()

In [ ]:
# Partition reduction: Remove a partition transform
# This stops future writes from using that partition, but old files remain

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    DROP PARTITION FIELD days(tpep_pickup_datetime)
""")

In [ ]:
# Partition addition: Add multiple partition transforms
spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    ADD PARTITION FIELD months(tpep_pickup_datetime)
""")

In [ ]:
# Insert new data - it will use the new partition spec
spark.sql("""
    INSERT INTO polaris.taxi.trips_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-08.parquet`
""")

In [ ]:
# View partitions now - mix of unpartitioned and day-partitioned data and month partitioned
spark.sql("""
    SELECT partition FROM polaris.taxi.trips_evolving.partitions
""").show()

In [ ]:
# Demonstrate automatic predicate transformation
# Query with timestamp filter - Iceberg automatically converts to partition filter

spark.sql("""
    SELECT 
        COUNT(*) as trip_count,
        MIN(tpep_pickup_datetime) as first_trip,
        MAX(tpep_pickup_datetime) as last_trip
    FROM polaris.taxi.trips_evolving
    WHERE tpep_pickup_datetime >= '2023-08-01 00:00:00'
      AND tpep_pickup_datetime < '2023-08-02 00:00:00'
""").show()